# End-to-End Evaluator Walkthrough: RAD-DINO + VinDr-CXR

This notebook wires every piece of `radharmony.evaluator` together on a single canonical example: running a **linear probe** on top of frozen **`microsoft/rad-dino`** features over **VinDr-CXR**.

Components exercised:

| Piece | What it does |
|---|---|
| `EncoderPreprocessTransform.from_huggingface` | Adapts the HF `AutoImageProcessor` into a MONAI `MapTransform` (reads the DICOM, resizes, normalizes with RAD-DINO's recipe). |
| `RadiologyEncoderTransform` | Fluent builder that layers RadHarmony augmentations (flip, intensity jitter, ...) on top of the encoder's preprocessing. |
| `ImageEncoderWrapper.from_huggingface` | Wraps the HF model so `.to(device)` / `.parameters()` propagate and the forward returns `Tensor[B, D]`. |
| `LinearProbeEvaluator` | Per-label logistic regression on frozen features, k-fold or fixed-split with `n_train` sweep. |

**Requirements**
- `radharmony` installed (`pip install -e .`) with `scikit-learn` and `transformers`.
- A VinDr-CXR DICOM dump on disk. Point `VINDR_TRAIN_DIR` / `VINDR_TEST_DIR` below at your copy.
- A CUDA GPU. CPU works but is slow for the CLS-feature extraction step.

## 1. Configuration

Edit these paths to point at your VinDr-CXR copy. `LABEL_SUBSET` keeps the probe fast — drop the list to use all 28 labels.

In [ ]:
VINDR_TRAIN_DIR = (
    "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/train/"
)
VINDR_TEST_DIR = (
    "/path/to/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
)
CACHE_DIR = "/path/to/cache/"
EMBEDDING_CACHE = "/path/to/cache/raddino_embeddings"

# Quick-run knobs — raise these for a real evaluation.
LABEL_SUBSET = [
    "lung_opacity",
    "cardiomegaly",
    "pleural_thickening",
    "aortic_enlargement",
    "pulmonary_fibrosis",
    "tuberculosis",
    "pleural_effusion",
]
N_FOLDS = 5
N_TRAIN_SAMPLES = [1500, 15000]
BATCH_SIZE = 32
DEVICE = "cuda:7"

## 2. Build the transform

`EncoderPreprocessTransform.from_huggingface` handles **loading + resize + normalization** with RAD-DINO's own processor. `reader="monai"` is the right choice for DICOM (uses MONAI's `LoadImage` + the `(C, W, H) → (C, H, W)` transpose).

`RadiologyEncoderTransform` wraps that preprocess step inside a RadHarmony fluent builder, so the standard `.with_flip` / `.with_intensity_jitter` augmentation hooks still work.

In [ ]:
from transformers import AutoImageProcessor

from radharmony.evaluator import (
    EncoderPreprocessTransform,
    RadiologyEncoderTransform,
)

processor = AutoImageProcessor.from_pretrained(
    "microsoft/rad-dino",
    size={"shortest_edge": 518},
)
pp = EncoderPreprocessTransform.from_huggingface(
    keys=["img"],
    processor=processor,
    reader="monai",
)

transform = (
    RadiologyEncoderTransform(preprocess=pp, output_keys={"img", "cls"})
    .with_flip(spatial_axis=1)
    .with_intensity_jitter(shift=0.05)
    .get_transform()
)

## 3. Load the dataset

Pass the encoder-aware transform into `VinDrCXRTrainDataset`. `output_cls=True` is required — the evaluator reads `batch["cls"]` to build the per-label targets.

In [ ]:
from radharmony.dataset import VinDrCXRTrainDataset

train_ds = VinDrCXRTrainDataset(
    base_image_dir=VINDR_TRAIN_DIR,
    transform=transform,
    cache_dir=CACHE_DIR,
    output_cls=True,
)
df = train_ds.get_harmonized_df()
print(f"train_ds: {len(df)} images, {len(train_ds.LABEL_COLS)} labels")

## 4. Wrap the encoder

`ImageEncoderWrapper.from_huggingface` routes through `model(pixel_values=x)` and CLS-pools `last_hidden_state`. The wrapper is a real `nn.Module`, so `.to(device)` / `.parameters()` propagate — this matters for `FinetuneEvaluator` (not used here) and for any checkpoint saving you do later.

In [ ]:
from transformers import AutoModel

from radharmony.evaluator import ImageEncoderWrapper

encoder = ImageEncoderWrapper.from_huggingface(
    AutoModel.from_pretrained("microsoft/rad-dino"),
    pool="cls",
).to(DEVICE)
print(f"encoder: {type(encoder.model).__name__}")

## 5. k-fold linear probe

`dataset=` activates k-fold mode. Embeddings are extracted once and cached at `EMBEDDING_CACHE`, then `KFold` splits the feature matrix `n_folds` ways and `n_train_samples` further subsamples each fold's training indices.

In [ ]:
from radharmony.evaluator import LinearProbeEvaluator

probe = LinearProbeEvaluator(
    image_encoder=encoder,
    dataset=train_ds,
    labels=LABEL_SUBSET,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)
df = probe.evaluate()
df.head()

In [ ]:
df

## 6. Per-label summary

`save_results` writes the raw rows (`results.csv`) and an aggregated summary (`results_summary.csv`, mean/std/95%-CI per label × `n_train`). Inspect the summary directly below.

In [ ]:
import pandas as pd

out_dir = probe.save_results(df, output_dir="./eval_out/raddino_kfold")
summary = pd.read_csv(f"{out_dir}/results_summary.csv")
summary[
    [
        "label",
        "n_train",
        "auroc_mean",
        "auroc_std",
        "auprc_mean",
        "auprc_std",
        "f1_mean",
        "f1_std",
    ]
]

## 7. Fixed-split mode (optional)

Swap the `dataset=` argument for `train_dataset=` + `test_dataset=` to enter fixed-split mode. Variance now comes from two sources: `n_seeds` (different train subsamples) × `n_bootstrap` (test-row resamples). The output schema is identical so the same `save_results` / `summary` code works downstream.

In [ ]:
from radharmony.dataset import VinDrCXRTestDataset

test_ds = VinDrCXRTestDataset(
    base_image_dir=VINDR_TEST_DIR,
    transform=transform,
    cache_dir=CACHE_DIR,
    output_cls=True,
)

probe_fixed = LinearProbeEvaluator(
    image_encoder=encoder,
    train_dataset=train_ds,
    test_dataset=test_ds,
    labels=LABEL_SUBSET,
    n_train_samples=[1500],
    n_seeds=5,
    n_bootstrap=1,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    embedding_cache=EMBEDDING_CACHE,
)
df_fixed = probe_fixed.evaluate()
df_fixed.groupby("label")[["auroc", "auprc", "f1"]].agg(["mean", "std"])

In [ ]:
=- 